# Praat Scripts (日本語版) - 区間継続時間の計算 (Google Colab版) 🎙️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okawawaka/praat-scripts-ja/blob/main/notebooks/calculate_segment_durations.ipynb)

本ノートブックは、[praat-scripts-ja](https://github.com/okawawaka/praat-scripts-ja) リポジトリの `scripts/acoustic-analysis/calculate_segment_durations.praat` と**完全に同一のアルゴリズム・数値結果**を Google Colab 上で再現するツールです。

PCに Praat をインストールしていなくても、ブラウザ上のファイル選択ダイアログから `.TextGrid` ファイルを選択するだけで、ミリ秒単位の継続時間を一括集計し、TSV ファイルとしてダウンロードできます。

### Step 1: 必要なライブラリのインストール
Praat の計算エンジン（`praat-parselmouth`）および `textgrid` ライブラリをインストールします。

In [ ]:
# 必要なライブラリのインストール
!pip install -q praat-parselmouth textgrid pandas

### Step 2: 分析パラメータの設定
右側のフォームから対象の Tier 番号や空白除外の設定を選択してください。

In [ ]:
# @title 【パラメータ設定】 { run: "auto" }
target_tier = 1 # @param {type:"integer"}
skip_empty_intervals = False # @param {type:"boolean"}

print(f"設定完了: 対象Tier = {target_tier}, 空白ラベル除外 = {skip_empty_intervals}")

### Step 3: ファイル選択ダイアログから TextGrid を読み込む
下のセルを実行すると、**ファイル選択ボタン** が表示されます。
お手元の PC から分析したい `.TextGrid` ファイル（複数選択可能）を選んでください。

In [ ]:
import io
import os
import tempfile
import textgrid
import pandas as pd
from google.colab import files

print("▼ 下の『ファイル選択』ボタンをクリックして、.TextGrid ファイルを選択してください（複数選択可）：")
uploaded = files.upload()

if not uploaded:
    print("ファイルが選択されませんでした。")
else:
    all_rows = []

    for filename, content in uploaded.items():
        if not filename.endswith(".TextGrid") and not filename.endswith(".textgrid"):
            print(f"スキップ（TextGridではありません）: {filename}")
            continue

        base_name = os.path.splitext(filename)[0]

        # 一時ファイルに保存して textgrid ライブラリで読み込み
        with tempfile.NamedTemporaryFile(suffix=".TextGrid", delete=False) as tmp:
            tmp.write(content)
            tmp_path = tmp.name

        try:
            tg = textgrid.TextGrid.fromFile(tmp_path)
            num_tiers = len(tg.tiers)

            if target_tier <= num_tiers:
                tier = tg.tiers[target_tier - 1] # 1-indexed -> 0-indexed

                if isinstance(tier, textgrid.IntervalTier):
                    for idx, interval in enumerate(tier, start=1):
                        label = interval.mark.strip()
                        start_t = interval.minTime
                        end_t = interval.maxTime
                        dur_s = end_t - start_t
                        dur_ms = dur_s * 1000.0

                        if skip_empty_intervals and label == "":
                            continue

                        all_rows.append({
                            "ObjectName": base_name,
                            "IntervalIndex": idx,
                            "Label": label,
                            "StartTime_s": f"{start_t:.4f}",
                            "EndTime_s": f"{end_t:.4f}",
                            "Duration_s": f"{dur_s:.4f}",
                            "Duration_ms": f"{dur_ms:.2f}"
                        })
                else:
                    print(f"警告: {filename} の Tier {target_tier} はインターバル段ではありません。")
            else:
                print(f"警告: {filename} には Tier {target_tier} が存在しません（総Tier数: {num_tiers}）。")
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

    if all_rows:
        df_results = pd.DataFrame(all_rows)
        print(f"\n集計完了: 合計 {len(df_results)} 件の区間データを集計しました。")
    else:
        df_results = pd.DataFrame()
        print("\n集計対象のデータがありませんでした。")

### Step 4: 集計結果の確認（Praatと同一形式の見出し）
Praatの Info ウィンドウや TSV ファイルと全く同一のフォーマットで表示されます。

In [ ]:
# 表形式でプレビュー表示
if 'df_results' in locals() and not df_results.empty:
    display(df_results)
else:
    print("表示するデータがありません。Step 3 でファイルをアップロードしてください。")

### Step 5: TSVファイルのダウンロード
集計結果を `duration_results.tsv` として保存し、PCにダウンロードします。
Excel や R などでそのまま読み込めます。

In [ ]:
if 'df_results' in locals() and not df_results.empty:
    out_tsv_path = "duration_results.tsv"
    df_results.to_csv(out_tsv_path, sep="\t", index=False, encoding="utf-8-sig")
    print(f"{out_tsv_path} を作成しました。ダウンロードを開始します...")
    files.download(out_tsv_path)
else:
    print("ダウンロード対象のデータがありません。")